<a href="https://colab.research.google.com/github/Ritiha-VS/AI-based-Forest-Fire-Detection/blob/main/TrainedModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from google.colab import drive
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report
)
import seaborn as sns
import matplotlib.pyplot as plt
import zipfile
import os

drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/test.zip"
extract_path = "/content/test_data"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
model_path = "/content/drive/MyDrive/tiny_fire_model.h5"
model = tf.keras.models.load_model(
    model_path,
    compile=False
)
size_bytes = os.path.getsize(model_path)

size_mb = size_bytes / (1024 * 1024)
size_kb = size_bytes / 1024
print(f"\nModel Size: {size_mb:.2f} MB")
print(f"Model Size: {size_kb:.2f} KB")
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/test_data/test",
    image_size=(128,128),
    batch_size=32,
    shuffle=False
)
test_dataset = test_dataset.map(lambda x, y: (x / 255.0, y))

y_true = np.concatenate([y for x, y in test_dataset], axis=0)
y_pred = model.predict(test_dataset)
y_pred = (y_pred > 0.5).astype(int).flatten()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"\nAccuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=['No Fire', 'Fire']
))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Fire', 'Fire'],
    yticklabels=['No Fire', 'Fire']
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
!pip install tensorflow opencv-python matplotlib numpy -q

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from google.colab import files

model_path = "/content/drive/MyDrive/tiny_fire_model.h5"
model = load_model(model_path)

print("Model loaded successfully")
print("Upload image for fire detection")
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
img = image.load_img(img_path, target_size=(128,128))
img_array = image.img_to_array(img)
img_norm = img_array / 255.0
img_norm = np.expand_dims(img_norm, axis=0)
prediction = model.predict(img_norm)
pred = prediction[0][0]
print("Prediction value:", pred)
if pred < 0.5:
    label = "Fire Detected"
    fire_detected = True
else:
    label = "No Fire"
    fire_detected = False

print(label)
img_cv = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

hsv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)
lower_fire = np.array([0,120,150])
upper_fire = np.array([35,255,255])
mask = cv2.inRange(hsv, lower_fire, upper_fire)
kernel = np.ones((7,7),np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel)
outline_img = img_cv.copy()
contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

for c in contours:
    if fire_detected and cv2.contourArea(c) > 500:
        x,y,w,h = cv2.boundingRect(c)
        cv2.rectangle(outline_img,(x,y),(x+w,y+h),(0,255,0),3)
outline_img = cv2.cvtColor(outline_img, cv2.COLOR_BGR2RGB)

heatmap = cv2.GaussianBlur(mask,(31,31),0)
heatmap = cv2.normalize(heatmap,None,0,255,cv2.NORM_MINMAX)
heatmap = np.uint8(heatmap)
heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

overlay = cv2.addWeighted(img_cv,0.6,heatmap_color,0.4,0)
overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(18,6))
plt.subplot(1,4,1)
plt.title("Original Image")
plt.imshow(img_rgb)
plt.axis("off")
plt.subplot(1,4,2)
plt.title(label)
plt.imshow(img_rgb)
plt.axis("off")
plt.subplot(1,4,3)
if fire_detected:
    plt.title("Fire Region Outline")
    plt.imshow(outline_img)
else:
    plt.title("No Fire Region")
    plt.imshow(img_rgb)
plt.axis("off")
plt.subplot(1,4,4)
plt.title("Fire Heatmap")
plt.imshow(overlay)
plt.axis("off")
plt.show()